# LOCA-PRAM — Negative-sample evaluation

Run the trained detector on **negative** assay images (no particles expected) to
characterize false-positive behavior at the production threshold and across a
sweep around it.

**Sampling.** Stratified: `IMAGES_PER_DEMO` images are randomly drawn from each
`demo_XXXX/` folder under every `*neg*` session. With 3 images / 12 demos / 2
sessions that's ~72 images — change `IMAGES_PER_DEMO = None` to run them all.

**Threshold strategy.** Each image is run through the model exactly once; the
sweep just re-thresholds the cached probability map, so it's effectively free.

**Outputs.**
- `negative_eval_results.csv` — one row per (image, threshold).
- `negative_eval_summary.png` — per-threshold detection-count distribution.


## 1. Imports + model architecture

Self-contained copy of the architecture from `LOCA_PRAM_main.ipynb` so this notebook can run standalone.

In [ ]:
import os
import glob
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import scipy.ndimage as ndi
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


In [ ]:
class conv_block(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.1, norm_groups=6, dilation=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, dilation, dilation=dilation, bias=True),
            nn.GroupNorm(norm_groups, out_channels),
            nn.ELU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, dilation, dilation=dilation, bias=True),
            nn.GroupNorm(norm_groups, out_channels),
            nn.ELU(),
        )
    def forward(self, x): return self.conv(x)


class up_conv(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.1, norm_groups=6):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(in_channels, out_channels, 3, 1, padding="same", bias=True),
        )
    def forward(self, x): return self.up(x)


class multi_head(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.1, norm_groups=6):
        super().__init__()
        self.multi_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, 1, padding="same", bias=True),
            nn.GroupNorm(norm_groups, in_channels),
            nn.ELU(),
            nn.Conv2d(in_channels, out_channels, 1, 1, padding="same", bias=True),
        )
    def forward(self, x): return self.multi_head(x)


class GaussianMixtureModel(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.out_channels_heads = (1, 3, 3, 1)
        self.num_channels = num_channels

        self.Conv1 = conv_block(num_channels, 36, norm_groups=6)
        self.Maxpool1 = nn.MaxPool2d(2, 2)
        self.Conv2 = conv_block(36, 72, norm_groups=6)
        self.Maxpool2 = nn.MaxPool2d(2, 2)
        self.Conv3 = conv_block(72, 144, norm_groups=6)
        self.Maxpool3 = nn.MaxPool2d(2, 2)
        self.Conv4 = nn.Sequential(
            conv_block(144, 288, norm_groups=6, dilation=4),
            conv_block(288, 288, norm_groups=6, dilation=8),
        )
        self.Up3 = up_conv(288, 144, norm_groups=6)
        self.Up_conv3 = conv_block(288, 144, norm_groups=6)
        self.Up2 = up_conv(144, 72, norm_groups=6)
        self.Up_conv2 = conv_block(144, 72, norm_groups=6)
        self.Up1 = up_conv(72, 36, norm_groups=6)
        self.Up_conv1 = conv_block(72, 36, norm_groups=6)
        self.dropout = nn.Dropout2d(p=0.3)
        self.sigma_eps = 0.001
        self.mt_heads = nn.ModuleList([
            multi_head(36, ch, norm_groups=6) for ch in self.out_channels_heads
        ])
        self.initialize_weights()
        nn.init.constant_(self.mt_heads[0].multi_head[-1].bias, -6.0)
        nn.init.zeros_(self.mt_heads[0].multi_head[-1].weight)

    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.GroupNorm):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x, training=False):
        x1 = self.Conv1(x)
        x2 = self.Conv2(self.Maxpool1(x1))
        x3 = self.Conv3(self.Maxpool2(x2))
        x4 = self.Conv4(self.Maxpool3(x3))
        d3 = self.Up3(x4); d3 = torch.cat((x3, d3), dim=1); d3 = self.Up_conv3(d3)
        d2 = self.Up2(d3); d2 = torch.cat((x2, d2), dim=1); d2 = self.Up_conv2(d2)
        d1 = self.Up1(d2); d1 = torch.cat((x1, d1), dim=1); d1 = self.Up_conv1(d1)
        d = [h(d1) for h in self.mt_heads]
        p = torch.sigmoid(torch.clamp(d[0], min=-20.0, max=20.0))
        pxyn_mean = d[1]
        pxyn_mean[:, [0, 1], ...] = torch.tanh(pxyn_mean[:, [0, 1], ...])
        pxyn_mean[:, [2], ...] = torch.sigmoid(pxyn_mean[:, [2], ...]) * 100
        pxy_std = torch.sigmoid(d[2]) * 50 + self.sigma_eps
        bg = d[3]
        return p, pxyn_mean, pxy_std, bg


## 2. Inference helpers

Tiled inference returning only the stitched probability map (we don't need sub-pixel coordinates for negative counting).

In [ ]:
def load_red_channel(path):
    img = np.asarray(Image.open(path)).astype(np.float32)
    if img.ndim == 3:
        img = img[:, :, 0]  # red channel
    return img


def infer_pmap(model, device, image, window_size=(512, 512)):
    """Tile -> per-tile normalize -> model -> stitch. Returns float32 p_full."""
    h, w = image.shape
    n_ty = h // window_size[0]
    n_tx = w // window_size[1]
    image = image[:n_ty * window_size[0], :n_tx * window_size[1]]

    tiles = []
    for ty in range(n_ty):
        for tx in range(n_tx):
            t = image[ty*window_size[0]:(ty+1)*window_size[0],
                      tx*window_size[1]:(tx+1)*window_size[1]]
            s = t.std() or 1.0
            tiles.append((t - t.mean()) / s)
    tiles_t = torch.from_numpy(np.array(tiles)).float().unsqueeze(1)

    p_chunks = []
    with torch.no_grad():
        for i in range(tiles_t.shape[0]):
            p_i, _, _, _ = model(tiles_t[i:i+1].to(device), training=False)
            p_chunks.append(p_i.cpu())
    p = torch.cat(p_chunks, dim=0).numpy()
    _, _, ph, pw = p.shape
    p_full = np.zeros((n_ty * ph, n_tx * pw), dtype=np.float32)
    for i in range(p.shape[0]):
        ty, tx = divmod(i, n_tx)
        p_full[ty*ph:(ty+1)*ph, tx*pw:(tx+1)*pw] = p[i, 0]
    return p_full


def count_clusters(p_map, p_threshold):
    binary = (p_map > p_threshold).astype(np.int32)
    _, num = ndi.label(binary)
    return num


## 3. Config

Edit paths/seeds/thresholds here. The threshold list matches the sweep range used in `LOCA_PRAM_main.ipynb`; `DEFAULT_THRESHOLD` is the production operating point.

In [ ]:
ASSAYS_GLOB = "assays/dose_response_buffer/*neg*"   # session folders
DEMO_GLOB = "demo_*"                                # cycle folders
IMAGE_EXT = ".jpg"

IMAGES_PER_DEMO = 3       # set to None to run every image in every demo
RANDOM_SEED = 0

MODEL_PATH = "final_v2.pth"
WINDOW_SIZE = (512, 512)

DEFAULT_THRESHOLD = 0.0021
SWEEP_THRESHOLDS = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008]
ALL_THRESHOLDS = sorted(set(SWEEP_THRESHOLDS + [DEFAULT_THRESHOLD]))

RESULTS_CSV = "negative_eval_results.csv"
SUMMARY_PNG = "negative_eval_summary.png"


## 4. Collect sampled images

In [ ]:
random.seed(RANDOM_SEED)

session_dirs = sorted(glob.glob(ASSAYS_GLOB))
print(f"Sessions found: {len(session_dirs)}")
for s in session_dirs:
    print(f"  {s}")

records = []  # list of dicts: session, demo, path
for sess in session_dirs:
    sess_name = os.path.basename(sess)
    demo_dirs = sorted(glob.glob(os.path.join(sess, DEMO_GLOB)))
    for demo in demo_dirs:
        demo_name = os.path.basename(demo)
        imgs = sorted(glob.glob(os.path.join(demo, f"*{IMAGE_EXT}")))
        if not imgs:
            continue
        chosen = imgs if IMAGES_PER_DEMO is None else random.sample(
            imgs, min(IMAGES_PER_DEMO, len(imgs)))
        for p in chosen:
            records.append({"session": sess_name, "demo": demo_name, "path": p})

print(f"\nTotal images to process: {len(records)}")
if records:
    print("First few:")
    for r in records[:5]:
        print(" ", r["session"], r["demo"], os.path.basename(r["path"]))


## 5. Load model

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = GaussianMixtureModel(num_channels=1)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device).eval()
print(f"Loaded {MODEL_PATH} on {device}")


## 6. Run batch inference

Progress and partial results stream to the CSV after every image, so a crash mid-run doesn't lose work.

In [ ]:
rows = []
csv_path = RESULTS_CSV

t0 = time.time()
for i, rec in enumerate(tqdm(records, desc="images")):
    try:
        img = load_red_channel(rec["path"])
        p_full = infer_pmap(model, device, img, window_size=WINDOW_SIZE)
        max_p = float(p_full.max())
        for thr in ALL_THRESHOLDS:
            n = int(count_clusters(p_full, thr))
            rows.append({
                "session": rec["session"],
                "demo": rec["demo"],
                "image": os.path.basename(rec["path"]),
                "path": rec["path"],
                "threshold": thr,
                "n_detections": n,
                "max_p": max_p,
                "image_h": img.shape[0],
                "image_w": img.shape[1],
            })
    except Exception as e:
        print(f"!! {rec['path']}: {e}")
        continue

    # incremental save every 5 images
    if (i + 1) % 5 == 0 or i == len(records) - 1:
        pd.DataFrame(rows).to_csv(csv_path, index=False)

print(f"\nDone in {time.time()-t0:.1f}s. Wrote {csv_path} ({len(rows)} rows)")
df = pd.DataFrame(rows)
df.head()


## 7. Summary

At the production threshold (`DEFAULT_THRESHOLD`), we expect detections per negative image to be sharply peaked near 0. The sweep confirms that nudging the threshold up/down doesn't drastically change that picture (i.e. negatives aren't *just* a threshold problem).

In [ ]:
df = pd.read_csv(RESULTS_CSV)

print("=== Detections per image at each threshold ===")
print(df.groupby("threshold")["n_detections"].describe()[["count", "mean", "50%", "max"]])

print(f"\n=== At DEFAULT_THRESHOLD={DEFAULT_THRESHOLD} ===")
dfd = df[df["threshold"] == DEFAULT_THRESHOLD]
print(dfd.groupby("session")["n_detections"].describe()[["count", "mean", "50%", "max"]])

print("\n=== Worst 10 images at default threshold ===")
print(dfd.sort_values("n_detections", ascending=False)
        .head(10)[["session", "demo", "image", "n_detections", "max_p"]]
        .to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: detection-count histogram at default threshold, split by session
dfd = df[df["threshold"] == DEFAULT_THRESHOLD]
sessions = sorted(dfd["session"].unique())
bins = np.arange(0, max(dfd["n_detections"].max() + 2, 10))
for s in sessions:
    axes[0].hist(dfd[dfd["session"] == s]["n_detections"], bins=bins,
                 alpha=0.6, label=s)
axes[0].set_xlabel("# detections per image")
axes[0].set_ylabel("# images")
axes[0].set_title(f"Negatives @ threshold={DEFAULT_THRESHOLD}")
axes[0].legend(fontsize=8)

# Right: mean detections vs threshold (sweep view)
sweep = df.groupby("threshold")["n_detections"].agg(["mean", "median", "max"]).reset_index()
axes[1].plot(sweep["threshold"], sweep["mean"], "o-", label="mean")
axes[1].plot(sweep["threshold"], sweep["median"], "s-", label="median")
axes[1].plot(sweep["threshold"], sweep["max"], "x--", label="max", alpha=0.5)
axes[1].axvline(DEFAULT_THRESHOLD, color="k", ls=":", label=f"default={DEFAULT_THRESHOLD}")
axes[1].set_xlabel("p_threshold")
axes[1].set_ylabel("# detections per image")
axes[1].set_yscale("symlog")
axes[1].set_title("Threshold sweep on negatives")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(SUMMARY_PNG, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved {SUMMARY_PNG}")


## 8. Inspect the top offenders (optional)

If a handful of images dominate the false-positive count, look at them directly to find out what's tripping the model (debris, hot pixels, autofluorescence).

In [ ]:
TOP_K = 6
dfd = df[df["threshold"] == DEFAULT_THRESHOLD].sort_values("n_detections", ascending=False)
top = dfd.head(TOP_K)

if len(top) == 0 or top["n_detections"].max() == 0:
    print("No detections at default threshold — nothing to inspect.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, (_, row) in zip(axes.flat, top.iterrows()):
        try:
            img = np.asarray(Image.open(row["path"]))
            ax.imshow(img if img.ndim == 3 else img, cmap=None if img.ndim == 3 else "gray")
            ax.set_title(f"{row['session']}/{row['demo']}\n{row['image']}\n"
                         f"n={row['n_detections']}, max_p={row['max_p']:.3f}",
                         fontsize=9)
        except Exception as e:
            ax.set_title(f"load failed: {e}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
